<div style="background-color: #f9f9f9; color: #333333; border: 1px solid #cccccc; padding: 10px; border-radius: 5px;">
Importing libraries.
</div>

In [ ]:
# ==========================================================
# Reduce warning messages from TensorFlow-GPU
# ==========================================================
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

In [ ]:
# ==========================================================
# Import important libraries
# ==========================================================
import pickle
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import (
    ModelCheckpoint,
    EarlyStopping,
    ReduceLROnPlateau,
    CSVLogger
)
from DeepLearning import build_dnn, build_cnn, build_cnn2


<div style="background-color: #f9f9f9; color: #333333; border: 1px solid #cccccc; padding: 10px; border-radius: 5px;">
Handling the dataset.
</div>


In [ ]:
# ==========================================================
# Load RML2016
# ==========================================================
def load_RML2016_dataset(dataset_path: str = "datasets/RML2016.pkl"):

    with open(dataset_path, "rb") as f:
        data = pickle.load(f, encoding="latin1")

    mods = sorted(list(set([key[0] for key in data.keys()])))
    snrs = sorted(list(set([key[1] for key in data.keys()])))

    mod_to_label = {mod: idx for idx, mod in enumerate(mods)}

    X_list = []
    y_list = []
    metadata = []

    for mod in mods:

        for snr in snrs:

            key = (mod, snr)

            if key not in data:
                continue

            samples = data[key]      # (1000, 2, 128)

            X_list.append(samples)

            labels = np.full(
                samples.shape[0],
                mod_to_label[mod],
                dtype=np.int64
            )

            y_list.append(labels)

            for i in range(samples.shape[0]):
                metadata.append({
                    "source": "RML2016",
                    "modulation": mod,
                    "snr": snr,
                    "class_id": mod_to_label[mod],
                    "sample_idx": i,
                })

    X = np.concatenate(X_list, axis=0).astype(np.float32)
    y = np.concatenate(y_list, axis=0).astype(np.int64)

    class_names = mods

    print("RML2016 dataset loaded.")
    print("X:", X.shape)
    print("y:", y.shape)
    print("Classes:", class_names)

    return X, y, metadata, class_names

In [ ]:
# ==========================================================
# General Dataset Interface
# ==========================================================
def get_dataset(RML2016_path: str = "datasets/RML2016.pkl"):

    X, y, metadata, class_names = load_RML2016_dataset(
        dataset_path=RML2016_path
    )

    # ======================================================
    # Reshape for VT-CNN2
    # ======================================================
    if X.ndim == 3:
        X = X[..., np.newaxis]      # (N, 2, 128, 1)

    X = X.astype(np.float32)

    num_classes = len(class_names)

    y_cat = to_categorical(
        y,
        num_classes=num_classes
    )

    print("\nPrepared for VT-CNN2:")
    print("X:", X.shape)
    print("y:", y.shape)
    print("y_cat:", y_cat.shape)
    print("num_classes:", num_classes)

    return X, y, y_cat, metadata, class_names

In [ ]:
# ==========================================================
# Train / Validation / Test Split
# ==========================================================
def split_dataset(X: np.ndarray,
                         y: np.ndarray,
                         y_cat: np.ndarray,
                         metadata: list[dict] | None = None,
                         random_state: int = 42):

    indices = np.arange(len(y))

    idx_train, idx_temp, _, y_temp_labels = train_test_split(
        indices,
        y,
        test_size=0.30,
        random_state=random_state,
        stratify=y
    )

    idx_val, idx_test = train_test_split(
        idx_temp,
        test_size=0.50,
        random_state=random_state,
        stratify=y_temp_labels
    )

    X_train = X[idx_train]
    X_val   = X[idx_val]
    X_test  = X[idx_test]

    y_train = y_cat[idx_train]
    y_val   = y_cat[idx_val]
    y_test  = y_cat[idx_test]

    metadata_train = None
    metadata_val   = None
    metadata_test  = None

    if metadata is not None:
        metadata_train = [metadata[i] for i in idx_train]
        metadata_val   = [metadata[i] for i in idx_val]
        metadata_test  = [metadata[i] for i in idx_test]

    print("Train:", X_train.shape, y_train.shape)
    print("Val  :", X_val.shape, y_val.shape)
    print("Test :", X_test.shape, y_test.shape)

    return (
        X_train, X_val, X_test,
        y_train, y_val, y_test,
        idx_train, idx_val, idx_test,
        metadata_train, metadata_val, metadata_test
    )

In [ ]:
# ==========================================================
# Load the dataset
# ==========================================================
X, y, y_cat, metadata, class_names = get_dataset(
    RML2016_path="datasets/RML2016.pkl"
)

# ==========================================================
# Keep examples with specific SNR range
# ==========================================================
snr_min = -20

snrs = np.array([m["snr"] for m in metadata])
keep_idx = np.where(snrs >= snr_min)[0]

X = X[keep_idx]
y = y[keep_idx]
metadata = [metadata[i] for i in keep_idx]

# IMPORTANT: rebuild y_cat AFTER filtering
num_classes = len(class_names)
y_cat = to_categorical(y, num_classes=num_classes)

X = X / np.sqrt(np.mean(X**2, axis=(1,2), keepdims=True))

num_classes = len(class_names)

(
    X_train, X_val, X_test,
    y_train, y_val, y_test,
    idx_train, idx_val, idx_test,
    metadata_train, metadata_val, metadata_test
) = split_dataset(
    X=X,
    y=y,
    y_cat=y_cat,
    metadata=metadata,
    random_state=42
)

<div style="background-color: #f9f9f9; color: #333333; border: 1px solid #cccccc; padding: 10px; border-radius: 5px;">
Training/validating/testing the Deep Learning model.
</div>


In [ ]:
# ==========================================================
# Training Configuration
# ==========================================================
model_name = "cnn2"   # "dnn", "cnn", or "cnn2".

results_dir = os.path.join(
    "results",
    model_name
)

os.makedirs(
    results_dir,
    exist_ok=True
)

batch_size = 1024
epochs = 100
learning_rate = 1e-3
dropout = 0.0

In [ ]:
# ==========================================================
# Build Model
# ==========================================================
builders = {
    "dnn": build_dnn,
    "cnn": build_cnn,
    "cnn2": build_cnn2,
}

model = builders[model_name](
    num_classes=num_classes,
    learning_rate=learning_rate,
    dropout=dropout,
    input_shape=(2, 128, 1)
)

model.summary()

# ==========================================================
# Callbacks
# ==========================================================
checkpoint_path = os.path.join(
    results_dir,
    f"best_{model_name}_model.keras"
)

callbacks = [
    ModelCheckpoint(
        filepath=checkpoint_path,
        monitor="val_loss",
        save_best_only=True,
        save_weights_only=False,
        verbose=1
    ),

    EarlyStopping(
        monitor="val_loss",
        patience=15,
        restore_best_weights=True,
        verbose=1
    ),

    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2,
        min_lr=1e-6,
        verbose=1
    ),

    CSVLogger(
        filename=os.path.join(
            results_dir,
            f"{model_name}_training_log.csv"
        ),
        append=True
    )
]

In [ ]:
# ==========================================================
# Training
# ==========================================================
history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=epochs,
    batch_size=batch_size,
    shuffle=True,
    callbacks=callbacks,
    verbose=1
)

In [ ]:
# ==========================================================
# Prediction
# ==========================================================
best_model = tf.keras.models.load_model(checkpoint_path)

y_pred_prob = best_model.predict(
    X_test,
    batch_size=batch_size,
    verbose=1
)

y_pred = np.argmax(y_pred_prob, axis=1)
y_true = np.argmax(y_test, axis=1)

In [ ]:
# ==========================================================
# Accuracy vs. SNR
# ==========================================================
snr_test = np.array([m["snr"] for m in metadata_test])

snr_values = np.array(sorted(np.unique(snr_test)))

accuracy_snr = []

print("\nAccuracy vs. SNR")
print("-" * 35)

for snr in snr_values:

    mask = snr_test == snr

    acc = np.mean(y_pred[mask] == y_true[mask])
    accuracy_snr.append(acc)

    print(f"SNR = {snr:>4} dB | Accuracy = {acc:.4f}")

In [ ]:
# ==========================================================
# Confusion Matrix
# ==========================================================
print(classification_report(
    y_true,
    y_pred,
    target_names=class_names
))

cm = confusion_matrix(y_true, y_pred)
print(cm)